# Buxter Drawing Playground

Интерактивный прогон pipeline фото+описание → FreeCAD → STL.

Требуется:
```bash
pip install -e '.[dev,notebook]'
```
и установленный FreeCAD (`freecadcmd --version`).

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

from buxter.config import load_settings
from buxter.vision import generate_script
from buxter.runner import run_freecad_script
from buxter.exporter import validate_artifacts

settings = load_settings()
print('model:', settings.model, '| out:', settings.output_dir)

In [ ]:
# Подстрой пути под свой сэмпл
photo_path = Path('../samples/bracket.jpg')
description_path = Path('../samples/bracket_description.txt')
out_dir = Path('../out/playground')
out_dir.mkdir(parents=True, exist_ok=True)

if photo_path.exists():
    display(Image.open(photo_path))
else:
    print(f'No photo at {photo_path} — будет работать без фото.')

In [ ]:
desc_widget = widgets.Textarea(
    value=description_path.read_text() if description_path.exists() else '',
    layout=widgets.Layout(width='100%', height='160px'),
)
model_widget = widgets.Dropdown(
    options=['opus', 'sonnet', 'haiku'], value=settings.model, description='Model:'
)
run_btn = widgets.Button(description='Generate', button_style='primary')
retry_btn = widgets.Button(description='Retry with edits', button_style='warning')
status = widgets.Output()


def _run(prior: bool):
    with status:
        clear_output()
        settings.model = model_widget.value
        prior_script = None
        stderr = None
        if prior:
            sp = out_dir / '_gen.py'
            lp = out_dir / 'run.log'
            if sp.exists():
                prior_script = sp.read_text()
            if lp.exists():
                stderr = lp.read_text()
        print('→ requesting script…')
        gen = generate_script(
            desc_widget.value,
            photo_path if photo_path.exists() else None,
            settings=settings,
            prior_script=prior_script,
            stderr=stderr,
        )
        print(f'→ running freecadcmd…')
        res = run_freecad_script(gen.script, out_dir, timeout=settings.run_timeout, freecad_cmd=settings.freecad_cmd)
        if not res.ok:
            print(f'FAILED exit={res.returncode}\n{res.stderr[-1500:]}')
            return
        art = validate_artifacts(res.stl_path, res.step_path)
        print(f'STL  = {art.stl} ({art.stl.stat().st_size} bytes)')
        if art.step:
            print(f'STEP = {art.step}')


run_btn.on_click(lambda _: _run(False))
retry_btn.on_click(lambda _: _run(True))
display(widgets.VBox([desc_widget, widgets.HBox([model_widget, run_btn, retry_btn]), status]))

In [ ]:
# Preview последней STL (требуется trimesh)
import trimesh
stl = out_dir / 'out.stl'
if stl.exists():
    mesh = trimesh.load(stl, force='mesh')
    print('triangles :', len(mesh.faces))
    print('bbox (mm) :', mesh.bounding_box.extents)
    print('volume    :', mesh.volume, 'mm³')
    print('watertight:', mesh.is_watertight)
    mesh.show()
else:
    print('Нет STL — запусти Generate выше.')